# TP2 — Analyse Factorielle des Correspondances (AFC)

**Cours :** Analyse Numérique & Statistique — GL3  
**Enseignant :** Pr. Adnène Arbi — INSAT  
**Étudiants :** Rayen Lagleg & Wissem Mekki

---

## Démarche en 8 étapes

Cette démarche suit rigoureusement la méthode du professeur : construction matricielle pas à pas, de la table de contingence brute jusqu'à la diagonalisation et la vérification croisée lignes/colonnes.

**Données :** Tableau de contingence 10×8 croisant 10 catégories socio-professionnelles avec 8 types d'hébergement en vacances.

---
## Imports et configuration

On importe uniquement **NumPy**, **Pandas** et **Matplotlib**. Toutes les étapes de l'AFC seront codées à la main pour que chaque calcul soit transparent et pédagogique.

In [ ]:
import numpy as np                        # calcul matriciel
import pandas as pd                       # affichage tabulaire
import matplotlib.pyplot as plt           # graphiques

pd.set_option('display.float_format', '{:.4f}'.format)   # 4 décimales
pd.set_option('display.max_columns', 20)                  # largeur max colonnes
np.set_printoptions(precision=4, suppress=True)           # pas de notation sci

---
## Étape 1 — Tableau de contingence brut $X$

Le tableau de contingence $X$ est une matrice $r \times c$ (ici $10 \times 8$) dont l'élément $x_{ij}$ représente le nombre d'individus appartenant simultanément à la modalité ligne $i$ et à la modalité colonne $j$ :

$$X = \bigl[x_{ij}\bigr]_{\substack{i=1\ldots r \\ j=1\ldots c}}, \qquad n = \sum_{i=1}^{r}\sum_{j=1}^{c} x_{ij}$$

L'effectif total $n$ est le nombre d'individus dans l'enquête. Il sert à normaliser toutes les fréquences dans les étapes suivantes.

In [ ]:
# ── Noms des modalités lignes (catégories socio-professionnelles) ──────
row_names = [
    'Agriculteur', 'Salariés',    'Patrons',     'Cadre sup',
    'Cadre moy',   'Employés',    'Ouvriers',    'Personnels',
    'Autres actif','Non actifs'
]

# ── Noms des modalités colonnes (types d'hébergement) ──────────────────
col_names = ['hotel', 'locat', 'propri', 'parent',
             'amis',  'tente', 'villag', 'divers']

# ── Tableau de contingence brut : x_ij = effectif observé ─────────────
X = np.array([
    [ 160,   28,   0,  321,  36,  141,  45,  65],   # Agriculteur
    [  35,   34,   1,  178,   8,    0,   4,   0],   # Salariés
    [ 700,  354, 229,  959, 185,  292, 119, 140],   # Patrons
    [ 961,  471, 633, 1580, 305,  360, 162, 148],   # Cadre sup
    [ 572,  537, 279, 1689, 206,  748, 155, 112],   # Cadre moy
    [ 441,  404, 166, 1079, 178,  434, 178,  92],   # Employés
    [ 783, 1114, 387, 4052, 497, 1464, 525, 387],   # Ouvriers
    [  65,   43,  21,  294,  79,   57,  18,   6],   # Personnels
    [  77,   60, 189,  839,  53,  124,  28,  53],   # Autres actif
    [ 741,  332, 327, 1789, 311,  236, 102, 102],   # Non actifs
], dtype=float)                   # dtype float pour éviter la division entière

# ── Effectif total n = somme de tous les éléments de X ─────────────────
n = X.sum()                       # scalaire : somme de la totalité du tableau

print(f'Dimensions de X : {X.shape[0]} lignes x {X.shape[1]} colonnes')
print(f'Effectif total n = {n:.0f}')
print()

# ── Affichage du tableau avec totaux marginaux ──────────────────────────
df_X = pd.DataFrame(X, index=row_names, columns=col_names)  # encapsulation Pandas
df_X_aff = df_X.copy()                          # copie pour ne pas modifier X
df_X_aff['TOTAL'] = df_X_aff.sum(axis=1)       # total par ligne
df_X_aff.loc['TOTAL'] = df_X_aff.sum(axis=0)  # total par colonne
print('Tableau de contingence brut X :')
df_X_aff

---
## Étape 2 — Tableau des fréquences relatives $P$

On divise chaque élément $x_{ij}$ par l'effectif total $n$ pour obtenir la fréquence relative correspondante :

$$P = \frac{X}{n}, \qquad p_{ij} = \frac{x_{ij}}{n}, \qquad \sum_{i,j} p_{ij} = 1$$

Le tableau $P$ est le point de départ de toute l'AFC : toutes les quantités suivantes (masses, profils, inertie) en sont dérivées.

In [ ]:
# ── Tableau des fréquences relatives : P = X / n ───────────────────────
P = X / n   # chaque cellule est divisée par l'effectif total

# ── Vérification : la somme de tous les éléments de P doit être 1 ──────
print(f'Somme de P = {P.sum():.10f}  (doit être 1.0)')
print()

# ── Affichage du tableau P ───────────────────────────────────────────────
df_P = pd.DataFrame(P, index=row_names, columns=col_names)  # encapsulation Pandas
print('Tableau des frequences relatives P = X / n :')
df_P

---
## Étape 3 — Masses marginales et matrices diagonales

Les **masses marginales** résument la distribution marginale du tableau $P$ :

$$f_i = \sum_{j=1}^{c} p_{ij} \quad \text{(masse de la ligne }i\text{)}, \qquad f_j = \sum_{i=1}^{r} p_{ij} \quad \text{(masse de la colonne }j\text{)}$$

On construit ensuite les quatre matrices diagonales de poids et leurs inverses :

$$D_I = \mathrm{diag}(f_1, \ldots, f_r) \in \mathbb{R}^{r \times r}, \qquad D_J = \mathrm{diag}(f_1, \ldots, f_c) \in \mathbb{R}^{c \times c}$$

$$D_I^{-1} = \mathrm{diag}\!\left(\frac{1}{f_1}, \ldots, \frac{1}{f_r}\right), \qquad D_J^{-1} = \mathrm{diag}\!\left(\frac{1}{f_1}, \ldots, \frac{1}{f_c}\right)$$

Ces matrices définissent les **métriques** (distances pondérées) dans les espaces respectifs des profils-lignes et des profils-colonnes.

In [ ]:
# ── Masses marginales lignes : fi[i] = somme sur j de p_ij ────────────
fi = P.sum(axis=1)   # vecteur de taille r = 10

# ── Masses marginales colonnes : fj[j] = somme sur i de p_ij ───────────
fj = P.sum(axis=0)   # vecteur de taille c = 8

# ── Vérification : les masses doivent sommer à 1 ────────────────────────
print(f'Somme des fi = {fi.sum():.10f}  (doit etre 1.0)')
print(f'Somme des fj = {fj.sum():.10f}  (doit etre 1.0)')
print()

# ── Matrices diagonales des masses lignes et colonnes ───────────────────
DI     = np.diag(fi)    # matrice diagonale (10x10) des masses lignes
DJ     = np.diag(fj)    # matrice diagonale (8x8) des masses colonnes
DI_inv = np.diag(1/fi)  # inverse de DI : diag(1/fi[i])  taille (10x10)
DJ_inv = np.diag(1/fj)  # inverse de DJ : diag(1/fj[j])  taille (8x8)

# ── Affichage de fi et fj sous forme de Series Pandas ───────────────────
print('fi (masses lignes) :')
display(pd.Series(fi, index=row_names, name='fi').to_frame().T)

print('\nfj (masses colonnes) :')
display(pd.Series(fj, index=col_names, name='fj').to_frame().T)

# ── Affichage des quatre matrices diagonales ─────────────────────────────
print('\nDI (10x10) =')
print(np.round(DI, 4))

print('\nDJ (8x8) =')
print(np.round(DJ, 4))

print('\nDI_inv (10x10) =')
print(np.round(DI_inv, 4))

print('\nDJ_inv (8x8) =')
print(np.round(DJ_inv, 4))

---
## Étape 4 — Profils-lignes $P_l$

Le profil-ligne de la ligne $i$ est la **distribution conditionnelle des colonnes sachant la ligne $i$** :

$$p_{j|i} = \frac{p_{ij}}{f_i} \qquad \Longrightarrow \qquad P_l = D_I^{-1}\, P \quad (\text{taille } r \times c)$$

Chaque ligne de $P_l$ somme à 1 :  $\displaystyle\sum_{j} (P_l)_{ij} = 1$.

Les profils-lignes représentent les points du nuage $N(I)$ dans $\mathbb{R}^c$. Comparer deux profils-lignes revient à comparer la répartition des hébergements de deux catégories socio-professionnelles.

In [ ]:
# ── Profils-lignes : Pl = DI_inv @ P ────────────────────────────────────
# Chaque ligne i de Pl est la distribution conditionnelle p_{j|i} = p_ij / fi[i]
Pl = DI_inv @ P   # produit matriciel (10x10) @ (10x8) = (10x8)

# ── Vérification : chaque ligne de Pl doit sommer à 1 ──────────────────
print('Somme de chaque ligne de Pl (doit etre 1.0) :')
print(np.round(Pl.sum(axis=1), 8))
print()

# ── Affichage des profils-lignes ─────────────────────────────────────────
df_Pl = pd.DataFrame(Pl, index=row_names, columns=col_names)  # encapsulation Pandas
print('Profils-lignes  Pl = DI_inv @ P :')
df_Pl

---
## Étape 5 — Profils-colonnes $P_c$

Le profil-colonne de la colonne $j$ est la **distribution conditionnelle des lignes sachant la colonne $j$** :

$$p_{i|j} = \frac{p_{ij}}{f_j} \qquad \Longrightarrow \qquad P_c = P\, D_J^{-1} \quad (\text{taille } r \times c)$$

Chaque colonne de $P_c$ somme à 1 :  $\displaystyle\sum_{i} (P_c)_{ij} = 1$.

Les profils-colonnes représentent les points du nuage $N(J)$ dans $\mathbb{R}^r$. Comparer deux profils-colonnes revient à comparer la composition socio-professionnelle de deux types d'hébergement.

In [ ]:
# ── Profils-colonnes : Pc = P @ DJ_inv ──────────────────────────────────
# Chaque colonne j de Pc est la distribution conditionnelle p_{i|j} = p_ij / fj[j]
Pc = P @ DJ_inv   # produit matriciel (10x8) @ (8x8) = (10x8)

# ── Vérification : chaque colonne de Pc doit sommer à 1 ────────────────
print('Somme de chaque colonne de Pc (doit etre 1.0) :')
print(np.round(Pc.sum(axis=0), 8))
print()

# ── Affichage des profils-colonnes ───────────────────────────────────────
df_Pc = pd.DataFrame(Pc, index=row_names, columns=col_names)  # encapsulation Pandas
print('Profils-colonnes  Pc = P @ DJ_inv :')
df_Pc

---
## Étape 6 — Centre de gravité du nuage $N(I)$

Le **centre de gravité** du nuage des profils-lignes $N(I)$ est le barycentre des profils-lignes pondéré par les masses $f_i$ :

$$G_l = \sum_{i=1}^{r} f_i\, \ell_i = f_i^\top\, P_l = f_j$$

où $\ell_i$ désigne le profil-ligne $i$ (la $i$-ème ligne de $P_l$). On montre que ce barycentre coïncide avec le vecteur des masses colonnes $f_j$.

**Pourquoi $G_l = f_j$ ?**  $\displaystyle\sum_i f_i\,(P_l)_{ij} = \sum_i f_i\,\frac{p_{ij}}{f_i} = \sum_i p_{ij} = f_j$. Le centre de gravité est le profil « moyen » que tout groupe socio-professionnel aurait si les lignes et les colonnes étaient indépendantes.

In [ ]:
# ── Centre de gravité du nuage des lignes N(I) : Gl = fj ───────────────
# Le barycentre pondéré des profils-lignes est le profil marginal des colonnes
Gl = fj   # vecteur de taille c = 8

# ── Vérification algébrique : fi @ Pl doit redonner fj ─────────────────
Gl_verif = fi @ Pl   # somme_i( fi[i] * Pl[i,:] ) -> vecteur (8,)
print('Gl = fj                :', np.round(Gl, 6))
print('fi @ Pl (verification) :', np.round(Gl_verif, 6))
print('Identiques             :', np.allclose(Gl, Gl_verif))
print()

# ── Affichage de Gl sous forme de Series Pandas ──────────────────────────
s_Gl = pd.Series(Gl, index=col_names, name='Gl (centre de gravite)')  # Series nommée
print('Centre de gravite Gl :')
s_Gl

---
## Étape 7 — Diagonalisation du nuage des lignes $N(I)$

On construit la matrice de variance-covariance du nuage $N(I)$, pondérée par les masses $f_i$ et exprimée dans la métrique $D_J^{-1}$ :

$$V = (P_l - G_l)^\top\, D_I\, (P_l - G_l) \in \mathbb{R}^{c \times c} \quad (\text{taille } 8 \times 8)$$

$$VM = V\, D_J^{-1} \in \mathbb{R}^{c \times c} \quad (\text{matrice à diagonaliser, taille } 8 \times 8)$$

Les **valeurs propres** $\lambda_k$ de $VM$ sont l'**inertie** portée par chaque axe factoriel, et les **vecteurs propres** donnent les directions principales dans l'espace des colonnes.

$$\sum_{k} \lambda_k = I_{\text{total}} = \sum_{i,j} \frac{(p_{ij} - f_i f_j)^2}{f_i f_j}$$

In [ ]:
# ── Centrage des profils-lignes : soustraction du centre de gravité Gl ──
centered_Pl = Pl - Gl   # broadcast : Gl (8,) soustrait de chaque ligne (10,8)

# ── Matrice V = (Pl-Gl)^T @ DI @ (Pl-Gl)  taille 8x8 ──────────────────
V = centered_Pl.T @ DI @ centered_Pl   # (8x10) @ (10x10) @ (10x8) = (8x8)

# ── Matrice à diagonaliser : VM = V @ DJ_inv ────────────────────────────
VM = V @ DJ_inv   # (8x8) @ (8x8) = (8x8)

# ── Décomposition spectrale : valeurs propres et vecteurs propres de VM ─
eigenvalues_raw, eigenvectors_raw = np.linalg.eig(VM)   # complexes potentiels

# ── Tri par ordre décroissant des valeurs propres (partie réelle) ────────
sort_idx     = np.argsort(eigenvalues_raw.real)[::-1]   # indices du tri
eigenvalues  = eigenvalues_raw.real[sort_idx]           # valeurs propres triées
eigenvectors = eigenvectors_raw.real[:, sort_idx]       # vecteurs propres triés

# ── Inertie totale, pourcentages et cumul ────────────────────────────────
total_inertia = eigenvalues.sum()                # somme = inertie totale
pct_inertia   = eigenvalues / total_inertia * 100   # % par axe
pct_cumul     = np.cumsum(pct_inertia)               # % cumulé

# ── Tableau récapitulatif des valeurs propres ────────────────────────────
df_eig = pd.DataFrame({
    'lambda_k': eigenvalues,          # inertie portée par l'axe k
    '% Inertie': pct_inertia,         # pourcentage d'inertie expliquée
    '% Cumule':  pct_cumul,           # pourcentage cumulé
}, index=[f'Axe {k+1}' for k in range(len(eigenvalues))])   # nommage
print('Valeurs propres, inertie et inertie cumulee :')
display(df_eig)

# ── Vecteurs propres de VM ───────────────────────────────────────────────
df_vect = pd.DataFrame(
    eigenvectors, index=col_names,
    columns=[f'v{k+1}' for k in range(len(eigenvalues))])   # colonnes = axes
print('\nVecteurs propres de VM (colonnes) :')
display(df_vect)

# ── Éboulis des valeurs propres (scree plot) ─────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))                       # figure et axe
x_pos = np.arange(len(eigenvalues))                         # positions sur x

ax.bar(x_pos, eigenvalues, color='steelblue',               # barres
       edgecolor='white', width=0.6, alpha=0.85)

ax.plot(x_pos, eigenvalues, marker='o', color='firebrick',  # courbe du coude
        linewidth=1.8, markersize=6, label='Valeurs propres $\lambda_k$')

for i, (vp, pct) in enumerate(zip(eigenvalues, pct_inertia)):   # annotations
    ax.text(i, vp + eigenvalues.max() * 0.015, f'{pct:.1f}%',
            ha='center', va='bottom', fontsize=8.5, color='navy')

ax.set_xticks(x_pos)                                         # ticks
ax.set_xticklabels([f'Axe {k+1}' for k in range(len(eigenvalues))],
                   rotation=30, ha='right')                  # étiquettes
ax.set_xlabel('Axes factoriels', fontsize=11)                # libellé x
ax.set_ylabel('Valeur propre $\lambda_k$', fontsize=11)    # libellé y
ax.set_title("Eboulis des valeurs propres — AFC\n"
             "Categories socio-professionnelles x Types d'hebergement",
             fontsize=12)                                     # titre
ax.legend(fontsize=9)                                        # légende
ax.grid(axis='y', linestyle='--', alpha=0.4)                # grille

plt.tight_layout()                                           # ajustement auto
plt.savefig('scree_plot_AFC.png', dpi=150, bbox_inches='tight')   # PNG
plt.show()                                                   # affichage

---
## Étape 8 — Diagonalisation du nuage des colonnes $N(J)$ et vérification

On répète l'analyse du côté des colonnes. Le centre de gravité du nuage $N(J)$ est $f_i$, et la matrice de variance-covariance dans la métrique $D_I^{-1}$ est :

$$V = (P_c - f_i)\, D_J\, (P_c - f_i)^\top \in \mathbb{R}^{r \times r} \quad (\text{taille } 10 \times 10)$$

$$VM = V\, D_I^{-1} \in \mathbb{R}^{r \times r} \quad (\text{matrice à diagonaliser, taille } 10 \times 10)$$

**Propriété fondamentale de l'AFC :** les valeurs propres non nulles de la matrice $10 \times 10$ sont **exactement les mêmes** que celles de la matrice $8 \times 8$ de l'étape 7. Les valeurs propres nulles supplémentaires de la matrice $10 \times 10$ correspondent à la dépendance linéaire des profils-colonnes (leur somme pondérée est $f_j$, une contrainte de rang).

In [ ]:
# ── Centrage des profils-colonnes : soustraction de fi de chaque ligne ──
# fi[:,None] transforme fi (10,) en colonne (10,1) pour le broadcast
centered_Pc = Pc - fi[:, None]   # (10x8) - (10x1) = (10x8) centré

# ── Matrice V = (Pc-fi) @ DJ @ (Pc-fi)^T  taille 10x10 ─────────────────
V = centered_Pc @ DJ @ centered_Pc.T   # (10x8) @ (8x8) @ (8x10) = (10x10)

# ── Matrice à diagonaliser : VM = V @ DI_inv ────────────────────────────
VM = V @ DI_inv   # (10x10) @ (10x10) = (10x10)

# ── Décomposition spectrale de VM (10x10) ───────────────────────────────
eigenvalues2_raw, _ = np.linalg.eig(VM)   # _ : vecteurs non utilisés ici

# ── Tri par ordre décroissant ────────────────────────────────────────────
sort_idx2    = np.argsort(eigenvalues2_raw.real)[::-1]   # indices du tri
eigenvalues2 = eigenvalues2_raw.real[sort_idx2]          # toutes les VP triées

# ── Conservation des valeurs propres non nulles (seuil > 1e-10) ─────────
eigenvalues2_nz = eigenvalues2[eigenvalues2 > 1e-10]    # filtrage des quasi-nulles

# ── Affichage de toutes les valeurs propres (10) ─────────────────────────
print('Toutes les valeurs propres de VM (10x10), triees :')
print(np.round(eigenvalues2, 8))
print()

# ── Valeurs propres non nulles du nuage colonnes ──────────────────────────
print('Valeurs propres non nulles -- nuage colonnes (etape 8) :')
print(np.round(eigenvalues2_nz, 8))
print()

# ── Valeurs propres non nulles du nuage lignes (étape 7) ─────────────────
eigenvalues_nz = eigenvalues[eigenvalues > 1e-10]   # filtrage étape 7
print('Valeurs propres non nulles -- nuage lignes   (etape 7) :')
print(np.round(eigenvalues_nz, 8))
print()

# ── Vérification de la propriété fondamentale de l'AFC ───────────────────
match = np.allclose(
    np.sort(eigenvalues2_nz)[::-1],   # tri décroissant côté colonnes
    np.sort(eigenvalues_nz)[::-1],    # tri décroissant côté lignes
    atol=1e-8                          # tolérance numérique
)
print(f'Valeurs propres identiques (etape 7 == etape 8) : {match}')
print()
print('Conclusion : la propriete fondamentale de l AFC est verifiee ---')
print('les VP non nulles du nuage lignes et du nuage colonnes coincident.')